<a href="https://colab.research.google.com/github/apache/beam/blob/master/examples/notebooks/get-started/iceberg-cdc-quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ###### Licensed to the Apache Software Foundation (ASF), Version 2.0 (the "License")

# Licensed to the Apache Software Foundation (ASF) under one
# or more contributor license agreements. See the NOTICE file
# distributed with this work for additional information
# regarding copyright ownership. The ASF licenses this file
# to you under the Apache License, Version 2.0 (the
# "License"); you may not use this file except in compliance
# with the License. You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing,
# software distributed under the License is distributed on an
# "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
# KIND, either express or implied. See the License for the
# specific language governing permissions and limitations
# under the License.

# Apache Iceberg CDC quickstart

Keep one Iceberg table in sync with another using change data capture (CDC), with three
[Beam SQL shells](https://beam.apache.org/documentation/dsls/sql/shell/). This notebook follows the
[Iceberg CDC quickstart](https://beam.apache.org/documentation/io/built-in/iceberg-cdc/).

## Set up

Pick where to run, and run the cell. The first run builds the shell, which takes a few minutes.

In [ ]:
#@title Set up
#@markdown **For Dataflow**, fill in a Google Cloud project with billing. Leave the bucket empty to use one named after the project. The cell creates the bucket and a Lakehouse catalog if they don't exist, and asks you to sign in.
run_on = "Locally"  #@param ["Locally", "Dataflow"]
project = ""  #@param {type:"string"}
bucket = ""  #@param {type:"string"}
region = "us-central1"  #@param {type:"string"}

# The rest of this cell starts three Beam SQL shells: the real shell, each in a terminal of its
# own. A %%sql cell types its statements into one of them, and shows what that shell printed.

import codecs
import fcntl
import glob
import html
import http.server
import importlib
import json
import os
import re
import shutil
import socket
import struct
import subprocess
import sys
import tempfile
import termios
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
import uuid

from IPython import get_ipython
from IPython.display import HTML, JSON, display

try:
  from google.colab import output as _colab_output
except ImportError:
  _colab_output = None

BEAM_VERSION = "2.79.0"
LOCAL_WAREHOUSE = "/tmp/beam-iceberg-cdc-quickstart"

_COLS, _ROWS = 72, 14
_SHELL_COLORS = {1: "#4285f4", 2: "#a142f4", 3: "#fa7b17"}
_FG, _BG = "#d4d4d4", "#1e1e1e"
_GREEN, _RED, _AMBER, _GRAY = "#1e8e3e", "#d93025", "#e37400", "#80868b"
_FONT = "ui-monospace, SFMono-Regular, Menlo, Consolas, 'Liberation Mono', monospace"
_PROMPT = re.compile(r"\d+: BeamSQL> ")
_CONTINUATION = re.compile(r"(?:\. )+> ")
_ALT_SCREEN = re.compile(r"\x1b\[\?(?:1049|1047|47)([hl])")
# The DEC line-drawing characters, which the live view draws its lines with.
_LINE_DRAWING = dict(zip("`afgjklmnopqrstuvwxyz{|}~", "◆▒°±┘┐┌└┼⎺⎻─⎼⎽├┤┴┬│≤≥π≠£·"))
# The terminal's colors, as VS Code's terminal shows them.
_PALETTE = {
    "black": "#000000", "red": "#cd3131", "green": "#0dbc79", "brown": "#e5e510",
    "blue": "#2472c8", "magenta": "#bc3fbc", "cyan": "#11a8cd", "white": "#e5e5e5",
    "brightblack": "#666666", "brightred": "#f14c4c", "brightgreen": "#23d18b",
    "brightbrown": "#f5f543", "brightblue": "#3b8eea", "brightmagenta": "#d670d6",
    "brightcyan": "#29b8db", "brightwhite": "#ffffff",
}

# Keep the shells from an earlier run of this cell, so that setting up again can stop them.
_shells = globals().get("_shells") or {}
_config = {"dataflow": None}
pyte = None  # The setup installs it.


class BeamSqlError(Exception):
  """An error that the notebook shows as just its message, in red."""

  def _render_traceback_(self):
    return ["\x1b[31m%s\x1b[0m" % self]


# Setup


def _setup(run_on, project="", bucket="", region=""):
  """Builds the Beam SQL shell, and starts three of them, connected to the catalog."""
  cloud = run_on == "Dataflow"
  if cloud:
    project = project.strip()
    if not project:
      raise BeamSqlError(
          "To run on Dataflow, fill in your Google Cloud project, and run this cell again.")
    bucket = bucket.strip().removeprefix("gs://").strip("/")
    bucket = bucket or re.sub(r"[^a-z0-9-]+", "-", project.lower()) + "-iceberg-cdc"
    region = region.strip() or "us-central1"
  _stop_shells()
  _config["dataflow"] = (
      {"project": project, "region": region, "bucket": bucket} if cloud else None)
  classpath = os.environ.get("BEAM_SQL_CLASSPATH")
  version = None
  if not classpath:
    version = _beam_version()
    classpath = os.path.expanduser(
        "~/.beam/cache/beam-%s_runner-%s_ios-iceberg.jar"
        % (version, "dataflow" if cloud else "direct"))
  if cloud:
    with _Step("Signing in to Google Cloud") as step:
      _sign_in(project)
      step.done("Signed in to Google Cloud")
    _set_up_google_cloud(project, bucket, region)
  java_home = _java()
  if version:
    _build(version, java_home, cloud, classpath)
  _install("pyte")
  if _colab_output is not None:
    _colab_output.register_callback("beam_sql.screen", _screen_callback)
    _serve()
  if not cloud:
    shutil.rmtree(LOCAL_WAREHOUSE, ignore_errors=True)
  _start_shells(java_home, classpath, _catalog_sql(cloud, project, bucket), cloud)


def _beam_version():
  """BEAM_VERSION if it's released, and its nightly snapshot until then."""
  if BEAM_VERSION.endswith("-SNAPSHOT"):
    return BEAM_VERSION
  url = (
      "https://repo1.maven.org/maven2/org/apache/beam/beam-sdks-java-core/{0}/"
      "beam-sdks-java-core-{0}.pom".format(BEAM_VERSION))
  try:
    urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=30).close()
    return BEAM_VERSION
  except urllib.error.HTTPError as e:
    if e.code == 404:
      return BEAM_VERSION + "-SNAPSHOT"
    raise


def _sign_in(project):
  os.environ["GOOGLE_CLOUD_PROJECT"] = project
  if _colab_output is not None:
    from google.colab import auth
    auth.authenticate_user(project_id=project)


def _set_up_google_cloud(project, bucket, region):
  """Creates the bucket and the catalog if they're missing, and gives Dataflow access."""
  if not shutil.which("gcloud"):
    raise BeamSqlError("Setting up Google Cloud needs the gcloud CLI. Install it, and try again.")
  with _Step("Turning on the Lakehouse and Dataflow APIs") as step:
    _gcloud("services", "enable", "biglake.googleapis.com", "dataflow.googleapis.com",
            "--project", project)
    step.done("The Lakehouse and Dataflow APIs are on")
  with _Step("Looking for bucket gs://%s" % bucket) as step:
    if _gcloud("storage", "buckets", "describe", "gs://" + bucket, check=False).returncode:
      step.update("Creating bucket gs://%s" % bucket)
      _gcloud("storage", "buckets", "create", "gs://" + bucket, "--project", project,
              "--location", region, "--uniform-bucket-level-access")
      step.done("Created bucket gs://%s" % bucket)
    else:
      step.done("Using bucket gs://%s" % bucket)
  catalog = ["biglake", "iceberg", "catalogs"]
  with _Step("Looking for Lakehouse catalog %s" % bucket) as step:
    described = _gcloud(*catalog, "describe", bucket, "--project", project, "--format=json",
                        check=False)
    if described.returncode:
      step.update("Creating Lakehouse catalog %s" % bucket)
      _gcloud(*catalog, "create", bucket, "--project", project, "--catalog-type=gcs-bucket",
              "--credential-mode=vended-credentials")
      described = _gcloud(*catalog, "describe", bucket, "--project", project, "--format=json")
      step.done("Created Lakehouse catalog %s" % bucket)
    else:
      step.done("Using Lakehouse catalog %s" % bucket)
  with _Step("Letting the catalog and Dataflow's workers use the bucket") as step:
    problems = []

    def grant(*args):
      result = _gcloud(*args, check=False)
      if result.returncode:
        problems.append(result.stderr.strip().splitlines()[-1])

    # The catalog hands out short-lived credentials for the bucket, made with its own account.
    catalog_account = json.loads(described.stdout).get("biglake-service-account")
    if catalog_account:
      grant("storage", "buckets", "add-iam-policy-binding", "gs://" + bucket,
            "--member=serviceAccount:" + catalog_account, "--role=roles/storage.objectUser")
    else:
      problems.append("Couldn't find the catalog's service account, to let it use the bucket.")
    # Dataflow's workers run as the project's Compute Engine default service account.
    number = _gcloud("projects", "describe", project, "--format=value(projectNumber)").stdout
    worker = "serviceAccount:%s-compute@developer.gserviceaccount.com" % number.strip()
    for role in ("roles/dataflow.worker", "roles/biglake.editor",
                 "roles/serviceusage.serviceUsageConsumer"):
      grant("projects", "add-iam-policy-binding", project, "--member=" + worker,
            "--role=" + role, "--condition=None")
    grant("storage", "buckets", "add-iam-policy-binding", "gs://" + bucket,
          "--member=" + worker, "--role=roles/storage.objectAdmin")
    if problems:
      step.warn(
          "Couldn't grant every role. If the pipelines can't reach the catalog, ask a project "
          "owner to run this cell.\n" + "\n".join(problems))
    else:
      step.done("The catalog and Dataflow's workers can use the bucket")


def _gcloud(*args, check=True):
  result = subprocess.run(["gcloud", *args, "--quiet"], capture_output=True, text=True)
  if check and result.returncode:
    raise BeamSqlError("gcloud %s failed:\n%s" % (" ".join(args[:4]), result.stderr.strip()))
  return result


def _java():
  """Returns the home of a JDK that runs Beam SQL, Java 17 to 23, installing one if needed."""
  with _Step("Looking for Java") as step:
    javas = [os.path.join(os.environ.get("JAVA_HOME", "/nonexistent"), "bin", "java"),
             shutil.which("java") or "/nonexistent"]
    javas += sorted(glob.glob("/usr/lib/jvm/*/bin/java"), reverse=True)
    homes = [os.path.dirname(os.path.dirname(os.path.realpath(j)))
             for j in javas if os.path.exists(j)]
    for home in dict.fromkeys(homes):
      version = _java_version(home)
      # Java 24 and later can't run the Hadoop catalog.
      if version and 17 <= version <= 23:
        step.done("Java %d" % version)
        return home
    if os.geteuid() != 0 or not shutil.which("apt-get"):
      raise BeamSqlError(
          "Beam SQL needs Java 17 to 23. Install it, point JAVA_HOME at it, and try again.")
    step.update("Installing Java 21")
    env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
    for args in (["update"], ["install", "-y", "openjdk-21-jdk-headless"]):
      result = subprocess.run(["apt-get", "-qq", *args], capture_output=True, text=True, env=env)
      if result.returncode:
        raise BeamSqlError("Couldn't install Java 21:\n" + _tail(result.stdout + result.stderr))
    for home in glob.glob("/usr/lib/jvm/java-21-openjdk-*"):
      if os.path.exists(os.path.join(home, "bin", "java")):
        step.done("Installed Java 21")
        return home
    raise BeamSqlError("Couldn't find Java 21 after installing it.")


def _java_version(home):
  try:
    result = subprocess.run([os.path.join(home, "bin", "java"), "-version"],
                            capture_output=True, text=True, timeout=60)
  except (OSError, subprocess.SubprocessError):
    return None
  match = re.search(r'version "(\d+)(?:\.(\d+))?', result.stderr)
  if not match:
    return None
  major = int(match.group(1))
  return int(match.group(2) or 0) if major == 1 else major


def _build(version, java_home, cloud, jar):
  """Builds the Beam SQL shell's JAR with beam-sql.sh, unless it's built already."""
  name = "Beam SQL shell %s" % version
  if os.path.exists(jar):
    _Step.show("✓", name, _GREEN)
    return
  note = " Beam %s isn't out yet, so this uses its nightly build." % BEAM_VERSION
  with _Step("Building the %s. The first time takes a few minutes.%s"
             % (name, note if version.endswith("-SNAPSHOT") else "")) as step:
    started = time.time()
    script = os.path.expanduser("~/.beam/beam-sql.sh")
    os.makedirs(os.path.dirname(script), exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/apache/beam/master/scripts/beam-sql.sh", script)
    os.chmod(script, 0o755)
    args = [script, "--version", version, "--io", "iceberg"]
    if cloud:
      args += ["--runner", "dataflow"]
    env = dict(os.environ, JAVA_HOME=java_home,
               PATH=os.path.join(java_home, "bin") + os.pathsep + os.environ["PATH"])
    try:
      # The script starts the shell once it's built, and "!quit" closes it.
      result = subprocess.run(args, input="!quit\n", capture_output=True, text=True, env=env,
                              cwd=os.path.expanduser("~"), timeout=1800)
      output = result.stdout + result.stderr
    except subprocess.TimeoutExpired as e:
      partial = e.output.decode(errors="replace") if isinstance(e.output, bytes) else e.output
      output = "Timed out after 30 minutes.\n%s" % (partial or "")
    if not os.path.exists(jar):
      raise BeamSqlError("Couldn't build the Beam SQL shell:\n" + _tail(output))
    step.done("Built the %s in %s" % (name, _duration(time.time() - started)))


def _install(package):
  try:
    module = importlib.import_module(package)
  except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
    importlib.invalidate_caches()
    module = importlib.import_module(package)
  globals()[package] = module


def _catalog_sql(cloud, project, bucket):
  if cloud:
    properties = [
        ("type", "rest"),
        ("uri", "https://biglake.googleapis.com/iceberg/v1/restcatalog"),
        ("warehouse", "gs://" + bucket),
        ("header.x-goog-user-project", project),
        ("rest.auth.type", "google"),
        ("io-impl", "org.apache.iceberg.gcp.gcs.GCSFileIO"),
        ("header.X-Iceberg-Access-Delegation", "vended-credentials"),
    ]
  else:
    properties = [("type", "hadoop"), ("warehouse", "file://" + LOCAL_WAREHOUSE)]
  return "CREATE CATALOG quickstart TYPE 'iceberg'\nPROPERTIES (\n%s\n)" % ",\n".join(
      "  '%s' = '%s'" % p for p in properties)


def _start_shells(java_home, classpath, catalog, cloud):
  """Starts the three shells, and connects each one to the catalog, like the quickstart does."""
  with _Step("Starting three Beam SQL shells") as step:
    for number in (1, 2, 3):
      _shells[number] = _Shell(number, os.path.join(java_home, "bin", "java"), classpath)
    statements = {number: [catalog, "USE DATABASE quickstart.shop"] for number in (1, 2, 3)}
    statements[1].insert(1, "CREATE DATABASE IF NOT EXISTS quickstart.shop")
    if cloud:
      # Start with empty tables: drop the ones an earlier run left in the catalog.
      statements[1] += ["DROP TABLE IF EXISTS orders", "DROP TABLE IF EXISTS orders_mirror"]
      dataflow = _config["dataflow"]
      statements[2] += [
          "SET runner = DataflowRunner",
          "SET project = '%s'" % dataflow["project"],
          "SET region = '%s'" % dataflow["region"],
          "SET tempLocation = 'gs://%s/tmp'" % dataflow["bucket"],
          "SET maxNumWorkers = 1",
      ]
    errors = {}

    def connect(number):
      shell = _shells[number]
      try:
        shell.wait_until_ready()
        for sql in statements[number]:
          start = shell.mark()
          shell.run(sql)
          error = shell.error_since(start)
          if error:
            errors[number] = error
            return
      except BeamSqlError as e:
        errors[number] = str(e)

    # Shell 1 creates the database, which the other two then use.
    connect(1)
    if 1 not in errors:
      threads = [threading.Thread(target=connect, args=(n,)) for n in (2, 3)]
      for thread in threads:
        thread.start()
      for thread in threads:
        thread.join()
    if not errors:
      step.done("Started three Beam SQL shells, each connected to the catalog")
  _display([_shells[n].window(_shells[n].transcript(0), small=True) for n in (1, 2, 3)])
  if errors:
    number, error = sorted(errors.items())[0]
    hint = ""
    if cloud and re.search(r"credential|401|403|unauthenticated", error, re.IGNORECASE):
      hint = ("\nIf the shells can't use your Google credentials, run "
              "!gcloud auth application-default login in a new cell, then this cell again.")
    raise BeamSqlError("Shell %d couldn't connect to the catalog: %s%s" % (number, error, hint))


def _stop_shells():
  """Stops what the shells are running, and closes them."""
  shells = list(_shells.values())
  for shell in shells:
    if shell.alive and shell.busy():
      shell.press_stop()
  for shell in shells:
    if shell.alive:
      shell.wait_for_prompt(60)
    shell.close()
  _shells.clear()


class _Step:
  """A line of setup progress, which gets a ✓ when the step is done."""

  def __init__(self, text):
    self.text, self.final, self.icon, self.color = text, None, "✓", _GREEN
    self.handle = display(HTML(self._html("…", text, _GRAY)), display_id=True)

  def __enter__(self):
    return self

  def update(self, text):
    self.text = text
    self.handle.update(HTML(self._html("…", text, _GRAY)))

  def done(self, text):
    self.final = text

  def warn(self, text):
    self.final, self.icon, self.color = text, "⚠", _AMBER

  def __exit__(self, kind, error, traceback):
    if error is None:
      self.handle.update(HTML(self._html(self.icon, self.final or self.text, self.color)))
    else:
      self.handle.update(HTML(self._html("✗", self.text, _RED)))

  @staticmethod
  def show(icon, text, color):
    display(HTML(_Step._html(icon, text, color)))

  @staticmethod
  def _html(icon, text, color):
    return ('<div style="font-family: %s; font-size: 13px; white-space: pre-wrap; margin: 2px 0">'
            '<span style="color: %s">%s</span> %s</div>' % (_FONT, color, icon, html.escape(text)))


# The shells


class _Terminal:
  """What a terminal shows, from what the shell writes to it.

  pyte keeps the screen. On top of it, this switches to the alternate screen that full-screen
  programs like the live view draw on, and draws the line-drawing characters, which pyte leaves out
  in UTF-8.
  """

  def __init__(self):
    self.main = pyte.HistoryScreen(_COLS, _ROWS, history=100000)
    self.alt = pyte.Screen(_COLS, _ROWS)
    self.streams = {False: pyte.Stream(self.main), True: pyte.Stream(self.alt)}
    self.decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    self.in_alt = False
    self.pending = ""
    self.line_drawing = [False, False]  # Whether G0 and G1 hold the line-drawing characters.
    self.shifted = False  # Whether the terminal draws with G1.
    self.version = 0

  def feed(self, data):
    text = self.pending + self.decoder.decode(data)
    self.pending = ""
    out, i, n = [], 0, len(text)
    while i < n:
      c = text[i]
      if c == "\x1b":
        if i + 1 >= n:
          self.pending = text[i:]
          break
        kind = text[i + 1]
        if kind in "()":
          if i + 2 >= n:
            self.pending = text[i:]
            break
          self.line_drawing[kind == ")"] = text[i + 2] == "0"
          i += 3
          continue
        if kind == "[":
          j = i + 2
          while j < n and not "@" <= text[j] <= "~":
            j += 1
          if j >= n:
            self.pending = text[i:]
            break
          switch = _ALT_SCREEN.fullmatch(text[i:j + 1])
          if switch:
            self._draw(out)
            entering = switch.group(1) == "h"
            if entering and not self.in_alt:
              self.alt.reset()
            self.in_alt = entering
          else:
            out.append(text[i:j + 1])
          i = j + 1
          continue
        if kind == "]":  # Like a window title, up to BEL or ST.
          ends = [e for e in (text.find("\x07", i), text.find("\x1b\\", i)) if e != -1]
          if not ends:
            self.pending = text[i:]
            break
          i = min(ends) + (1 if text[min(ends)] == "\x07" else 2)
          continue
        out.append(text[i:i + 2])
        i += 2
        continue
      if c in "\x0e\x0f":
        self.shifted = c == "\x0e"
      elif self.line_drawing[self.shifted]:
        out.append(_LINE_DRAWING.get(c, c))
      else:
        out.append(c)
      i += 1
    self._draw(out)
    self.version += 1

  def _draw(self, out):
    if out:
      self.streams[self.in_alt].feed("".join(out))
      out.clear()

  def mark(self):
    """Where the next line the shell writes goes, counting the lines that scrolled away."""
    return len(self.main.history.top) + self.main.cursor.y

  def lines(self, start):
    """The main screen's lines from `start` to the cursor, and where the cursor is."""
    lines = list(self.main.history.top) + [self.main.buffer[y] for y in range(self.main.lines)]
    end = self.mark() + 1
    return lines[start:end], self.main.cursor.x

  def screen(self):
    """The lines of the screen the terminal shows now."""
    screen = self.alt if self.in_alt else self.main
    return [screen.buffer[y] for y in range(screen.lines)]

  def typed(self):
    """The cursor's line up to the cursor: the prompt, and what's been typed after it."""
    return self.main.display[self.main.cursor.y][:self.main.cursor.x]

  def at_prompt(self):
    return not self.in_alt and _PROMPT.fullmatch(self.typed()) is not None

  def at_continuation(self):
    return not self.in_alt and _CONTINUATION.fullmatch(self.typed()) is not None


class _Shell:
  """A Beam SQL shell, running in a terminal of its own."""

  def __init__(self, number, java, classpath):
    self.number = number
    self.term = _Terminal()
    self.changed = threading.Condition()
    self.closed = False
    self.running = None  # The streaming statement the shell is running, if any.
    master, child = os.openpty()
    fcntl.ioctl(child, termios.TIOCSWINSZ, struct.pack("HHHH", _ROWS, _COLS, 0, 0))
    args = [
        java,
        # Show warnings, but not every log line, like the shell's own INFO logs.
        "-Djava.util.logging.config.file=" + _logging_config(),
        "-Dslf4j.internal.verbosity=ERROR",
        *os.environ.get("BEAM_SQL_JVM_FLAGS", "").split(),
        "-cp", classpath, "org.apache.beam.sdk.extensions.sql.jdbc.BeamSqlLine",
    ]
    if shutil.which("setsid"):
      # The terminal becomes the shell's own, so that Ctrl+C reaches it.
      args = ["setsid", "--ctty"] + args
    self.process = subprocess.Popen(
        args, stdin=child, stdout=child, stderr=child, close_fds=True,
        env=dict(os.environ, TERM="xterm-256color"))
    os.close(child)
    self.fd = master
    threading.Thread(target=self._read, daemon=True).start()

  def _read(self):
    while True:
      try:
        data = os.read(self.fd, 65536)
      except OSError:
        data = b""
      with self.changed:
        if data:
          self.term.feed(data)
        else:
          self.closed = True
        self.changed.notify_all()
      if not data:
        return

  @property
  def alive(self):
    return not self.closed and self.process.poll() is None

  def _write(self, text):
    os.write(self.fd, text.encode())

  def _wait(self, condition, timeout):
    end = time.time() + timeout
    with self.changed:
      while not condition():
        if self.closed:
          raise BeamSqlError("Shell %d stopped. Run the setup cell again." % self.number)
        left = end - time.time()
        if left <= 0:
          return False
        self.changed.wait(min(left, 0.5))
      return True

  def wait_until_ready(self):
    if not self._wait(self.term.at_prompt, 180):
      raise BeamSqlError("Shell %d didn't start." % self.number)

  def wait_for_prompt(self, timeout):
    return self._wait(self.term.at_prompt, timeout)

  def mark(self):
    with self.changed:
      return self.term.mark()

  def run(self, statement):
    """Types one statement, and waits for it. Returns "done", "live" or "running"."""
    lines = [line.rstrip() for line in statement.strip().splitlines() if line.strip()]
    lines[-1] += ";"
    for i, line in enumerate(lines):
      last_line = self.mark()
      self._write(line + "\r")
      if i < len(lines) - 1:
        # The shell asks for the rest of the statement, and then takes the next line.
        if not self._wait(lambda: self.term.mark() > last_line and self.term.at_continuation(),
                          30):
          raise BeamSqlError("Shell %d didn't take the statement." % self.number)
    finished = lambda: self.term.mark() > last_line and self.term.at_prompt()
    if not _is_streaming(statement):
      self._wait(finished, 3600)
      return "done" if finished() else "running"
    if re.match(r"\s*SELECT\b", statement, re.IGNORECASE):
      # The live view opens on the terminal's alternate screen.
      self._wait(lambda: self.term.in_alt or finished(), 600)
      return "live" if self.term.in_alt else "done"
    # A streaming INSERT keeps running, so the prompt doesn't come back.
    if self._wait(finished, 6):
      return "done"
    self.running = statement
    return "running"

  def busy(self):
    with self.changed:
      return not self.term.at_prompt()

  def press_stop(self):
    """Presses q to close the live view, or Ctrl+C to stop a running statement."""
    with self.changed:
      if self.term.in_alt:
        key = "q"
      elif self.term.at_continuation():
        key = ";\r"
      elif self.term.at_prompt():
        return
      else:
        key = "\x03"
    self.running = None
    self._write(key)

  def finish(self):
    """Stops what the shell is running, like before typing the next statement in a terminal."""
    if self.busy():
      self.press_stop()
      if not self.wait_for_prompt(300):
        raise BeamSqlError("Shell %d is still busy." % self.number)

  def transcript(self, start):
    with self.changed:
      lines, cursor = self.term.lines(start)
      return _render_lines(lines, cursor if self.term.at_prompt() else None)

  def error_since(self, start):
    with self.changed:
      lines, _ = self.term.lines(start)
    text = ""
    for line in lines:
      row = "".join(line[x].data for x in range(_COLS))
      # A full line goes on in the next one: the terminal wrapped it.
      text += row if len(row.rstrip()) == _COLS else row.rstrip() + "\n"
    match = re.search(r"^Error: (.*?)(?=\(state=|\n\d+: BeamSQL>|\Z)", text,
                      re.MULTILINE | re.DOTALL)
    return " ".join(match.group(1).split()) if match else None

  def badge(self):
    if self.term.in_alt:
      return "live view"
    if self.running:
      return "running on Dataflow" if _config["dataflow"] and self.number == 2 else "running"
    return ""

  def window(self, body, badge=None, full_width=False, small=False):
    link = ""
    if self.number == 2 and _config["dataflow"]:
      link = ('<a href="https://console.cloud.google.com/dataflow/jobs?project=%s" target="_blank"'
              ' style="margin-left: auto; color: #8ab4f8; font-weight: 400">Dataflow jobs ↗</a>'
              % urllib.parse.quote(_config["dataflow"]["project"]))
    return _window(self.number, body, self.badge() if badge is None else badge, link,
                   full_width, small)

  def live_window(self):
    """The terminal's screen, as wide as the terminal, like while the live view is open."""
    return self.window(self.screen_payload()["html"], full_width=True)

  def screen_payload(self):
    """The terminal's screen, for a live view that keeps updating."""
    with self.changed:
      body = _render_lines(self.term.screen(), None)
      return {"html": body, "badge": self.badge(), "version": self.term.version,
              "done": not self.term.in_alt or self.closed}

  def close(self):
    try:
      self.process.terminate()
      self.process.wait(5)
    except Exception:  # pylint: disable=broad-except
      self.process.kill()
    try:
      os.close(self.fd)
    except OSError:
      pass


def _logging_config():
  path = os.path.join(tempfile.gettempdir(), "beam-sql-notebook-logging.properties")
  with open(path, "w") as f:
    f.write("handlers = java.util.logging.ConsoleHandler\n"
            ".level = WARNING\n"
            "java.util.logging.ConsoleHandler.level = WARNING\n"
            "org.apache.hadoop.util.NativeCodeLoader.level = SEVERE\n")
  return path


def _shell(number):
  shell = _shells.get(number)
  if shell is None or not shell.alive:
    raise BeamSqlError("Shell %d isn't running. Run the setup cell at the top first." % number)
  return shell


def _split_statements(sql):
  """Splits `sql` at semicolons outside quotes, and drops -- comments."""
  statements, current, quote, i = [], [], None, 0
  while i < len(sql):
    c = sql[i]
    if quote:
      current.append(c)
      if c == quote:
        quote = None
    elif c in "'\"`":
      quote = c
      current.append(c)
    elif sql.startswith("--", i):
      end = sql.find("\n", i)
      i = len(sql) if end < 0 else end
      continue
    elif c == ";":
      statements.append("".join(current).strip())
      current = []
    else:
      current.append(c)
    i += 1
  statements.append("".join(current).strip())
  return [s for s in statements if s]


def _is_streaming(sql):
  return re.search(r"'streaming'\s*=\s*true", sql, re.IGNORECASE) is not None


def _sql_magic(line, cell):
  """%%sql shell1, %%sql shell2 or %%sql shell3: types the cell's SQL into that shell."""
  match = re.fullmatch(r"\s*shell\s*([123])\s*", line)
  if not match:
    raise BeamSqlError("Start the cell with %%sql shell1, %%sql shell2 or %%sql shell3.")
  shell = _shell(int(match.group(1)))
  shell.finish()
  start = shell.mark()
  state, wrote = "done", False
  for statement in _split_statements(cell):
    state = shell.run(statement)
    wrote = wrote or (statement.upper().startswith("INSERT") and not _is_streaming(statement))
    if state != "done":
      break
  if state == "live":
    _display([shell])
    return
  parts = [shell.window(shell.transcript(start))]
  # Next to a write, show the live view another shell has open, to watch the changes arrive.
  if wrote:
    parts += [s for s in _shells.values() if s is not shell and s.alive and s.term.in_alt]
  _display(parts)


def stop_all():
  """Closes the live view and stops the streaming pipeline, like q and Ctrl+C in a terminal."""
  busy = [shell for shell in _shells.values() if shell.alive and shell.busy()]
  if not busy:
    display(HTML(_note("Nothing is running.")))
    return
  starts = {shell.number: shell.mark() for shell in busy}
  for shell in busy:
    shell.press_stop()
  if _config["dataflow"] and any(shell.number == 2 for shell in busy):
    with _Step("Cancelling the Dataflow job, which takes a minute or two") as step:
      for shell in busy:
        shell.wait_for_prompt(300)
      step.done("Cancelled the Dataflow job")
  for shell in busy:
    shell.wait_for_prompt(300)
  _display([s.window(s.transcript(starts[s.number])) for s in busy])


# Showing the shells


def _display(parts):
  """Shows windows side by side. A part is a window's HTML, or a shell to show live."""
  if _colab_output is not None:
    display(HTML(_row([p if isinstance(p, str) else _live_html(p) for p in parts])))
    return

  # Outside Colab, the notebook updates the live view itself.
  def render():
    return _row([p if isinstance(p, str) else p.live_window()
                 for p in parts])

  live = [p for p in parts if not isinstance(p, str)]
  handle = display(HTML(render()), display_id=True)
  if not live or get_ipython() is None:
    return

  def refresh():
    while True:
      time.sleep(0.5)
      handle.update(HTML(render()))
      if all(shell.screen_payload()["done"] for shell in live):
        return

  threading.Thread(target=refresh, daemon=True).start()


def _row(windows):
  if len(windows) == 1:
    return windows[0]
  return ('<div style="display: flex; flex-wrap: wrap; align-items: flex-start">%s</div>'
          % "".join(windows))


def _live_html(shell):
  """In Colab, a shell's screen that keeps updating while the live view is open."""
  server = globals().get("_server")
  return (_LIVE.replace("__BOX__", "beam-sql-" + uuid.uuid4().hex)
          .replace("__SHELL__", str(shell.number))
          .replace("__PORT__", str(server.server_address[1]) if server else "null")
          .replace("__VERSION__", str(shell.term.version))
          .replace("__WINDOW__", shell.live_window()))


# In Colab, the live view shows the shell's screen in a frame that the kernel's web server serves,
# like TensorBoard does. If the frame can't load, it asks the kernel for the screen instead.
_LIVE = """
<div id="__BOX__">__WINDOW__</div>
<script>
(async () => {
  const box = document.getElementById("__BOX__");
  const shell = __SHELL__, port = __PORT__;
  let version = __VERSION__;
  try {
    if (port !== null && google.colab.kernel.accessAllowed) {
      const token = Math.random().toString(36).slice(2);
      const base = await google.colab.kernel.proxyPort(port, {cache: false});
      const frame = document.createElement("iframe");
      frame.src = new URL(`view.html?shell=${shell}&token=${token}`, base).toString();
      frame.style.cssText = "border: 0; width: 1px; height: 1px; position: absolute; opacity: 0";
      const ready = new Promise(resolve => {
        window.addEventListener("message", event => {
          const data = event.data || {};
          if (data.token !== token) return;
          frame.style.width = `${data.width}px`;
          frame.style.height = `${data.height}px`;
          resolve(true);
        });
        setTimeout(() => resolve(false), 10000);
      });
      box.after(frame);
      if (await ready) {
        box.remove();
        frame.style.position = "static";
        frame.style.opacity = "1";
        return;
      }
      frame.remove();
    }
  } catch (e) {}
  let failures = 0;
  for (;;) {
    await new Promise(resolve => setTimeout(resolve, 500));
    let data;
    try {
      const reply = await google.colab.kernel.invokeFunction("beam_sql.screen", [shell], {});
      data = reply.data["application/json"];
      failures = 0;
    } catch (e) {
      if (++failures < 5) continue;
      box.querySelector(".bsql-badge").textContent = "not updating: can't reach the kernel";
      return;
    }
    if (data.version !== version) {
      box.querySelector(".bsql-screen").innerHTML = data.html;
      box.querySelector(".bsql-badge").textContent = data.badge;
      version = data.version;
    }
    if (data.done) return;
  }
})();
</script>
"""

_VIEW = """<!doctype html>
<html><head><meta charset="utf-8">
<style>html, body { margin: 0; overflow: hidden; background: transparent; }</style></head>
<body>__WINDOW__
<script>
const shell = __SHELL__;
let version = __VERSION__;
const window_ = document.querySelector(".bsql-window");
window_.style.margin = "0";
parent.postMessage({token: new URLSearchParams(location.search).get("token"),
                    width: window_.offsetWidth, height: window_.offsetHeight}, "*");
(async () => {
  for (;;) {
    await new Promise(resolve => setTimeout(resolve, 500));
    try {
      const response = await fetch(`screen?shell=${shell}`, {cache: "no-store"});
      const data = await response.json();
      if (data.version !== version) {
        document.querySelector(".bsql-screen").innerHTML = data.html;
        document.querySelector(".bsql-badge").textContent = data.badge;
        version = data.version;
      }
      if (data.done) return;
    } catch (e) {}
  }
})();
</script></body></html>
"""


def _screen_callback(number):
  shell = _shells.get(number)
  if shell is None:
    return JSON({"html": "", "badge": "", "version": -1, "done": True})
  return JSON(shell.screen_payload())


class _Handler(http.server.BaseHTTPRequestHandler):

  def do_GET(self):  # pylint: disable=invalid-name
    url = urllib.parse.urlparse(self.path)
    query = urllib.parse.parse_qs(url.query)
    shell = _shells.get(int((query.get("shell") or ["0"])[0] or 0))
    if url.path.endswith("/screen"):
      payload = shell.screen_payload() if shell else {
          "html": "", "badge": "", "version": -1, "done": True}
      body, kind = json.dumps(payload).encode(), "application/json"
    elif url.path.endswith("/view.html") and shell:
      body = (_VIEW.replace("__SHELL__", str(shell.number))
              .replace("__VERSION__", str(shell.term.version))
              .replace("__WINDOW__", shell.live_window())).encode()
      kind = "text/html; charset=utf-8"
    else:
      self.send_error(404)
      return
    self.send_response(200)
    self.send_header("Content-Type", kind)
    self.send_header("Cache-Control", "no-store")
    # Don't save the answers in the notebook.
    self.send_header("x-colab-notebook-cache-control", "no-cache")
    self.end_headers()
    self.wfile.write(body)

  def log_message(self, *args):
    pass


def _serve():
  """Starts the web server for the live view, unless it's running already."""
  if globals().get("_server") is not None:
    return

  class Server(http.server.ThreadingHTTPServer):
    address_family = socket.AF_INET6
    daemon_threads = True

  try:
    server = Server(("::", 0), _Handler)
  except OSError:
    return  # The live view asks the kernel for the screen instead.
  threading.Thread(target=server.serve_forever, daemon=True).start()
  globals()["_server"] = server


def _window(number, body, badge="", extra="", full_width=False, small=False):
  """A terminal window. A live one is as wide as the terminal; others fit what they show."""
  width = "width: %dch" % _COLS if full_width else "min-width: 36ch"
  return (
      '<div class="bsql-window" style="display: inline-block; vertical-align: top;'
      ' margin: 4px 12px 8px 0; border-radius: 8px; overflow: hidden; background: %s;'
      ' border: 1px solid #3c3c3c; box-shadow: 0 1px 4px rgba(0, 0, 0, .3)">'
      '<div style="display: flex; align-items: center; gap: 8px; padding: 6px 12px;'
      ' background: #2b2b2b; border-bottom: 1px solid #3c3c3c; color: #e8e8e8;'
      ' font: 600 12px/1.3 system-ui, -apple-system, sans-serif">'
      '<span style="width: 9px; height: 9px; border-radius: 50%%; background: %s"></span>'
      'Shell %d<span class="bsql-badge" style="font-weight: 400; color: #9aa0a6">%s</span>%s</div>'
      '<pre class="bsql-screen" style="margin: 0; padding: 8px 12px; %s; box-sizing: content-box;'
      ' color: %s; background: %s; font: %s/1.35 %s; white-space: pre; overflow: hidden">%s</pre>'
      '</div>'
      % (_BG, _SHELL_COLORS[number], number, html.escape(badge), extra, width, _FG, _BG,
         "10.5px" if small else "12px", _FONT, body))


def _render_lines(lines, cursor_x):
  """HTML for terminal lines, in their colors. Draws the cursor on the last line if it's given."""
  out = []
  for index, line in enumerate(lines):
    cursor = cursor_x if index == len(lines) - 1 else None
    last = -1
    for x in range(_COLS):
      char = line[x]
      if char.data != " " or char.reverse or char.bg != "default":
        last = x
    if cursor is not None:
      last = max(last, cursor)
    spans, style, text = [], None, []
    for x in range(last + 1):
      char = line[x]
      fg = _PALETTE.get(char.fg, "#" + char.fg if re.fullmatch(r"[0-9a-f]{6}", char.fg) else _FG)
      bg = _PALETTE.get(char.bg, "#" + char.bg if re.fullmatch(r"[0-9a-f]{6}", char.bg) else None)
      if char.reverse:
        fg, bg = bg or _BG, fg
      if x == cursor:
        fg, bg = _BG, _FG
      current = (fg, bg, char.bold, char.underscore)
      if current != style:
        if text:
          spans.append(_span(style, "".join(text)))
        style, text = current, []
      text.append(char.data or "")
    if text:
      spans.append(_span(style, "".join(text)))
    out.append("".join(spans))
  return "\n".join(out)


def _span(style, text):
  fg, bg, bold, underline = style
  css = []
  if fg != _FG:
    css.append("color: %s" % fg)
  if bg:
    css.append("background: %s" % bg)
  if bold:
    css.append("font-weight: 700")
  if underline:
    css.append("text-decoration: underline")
  text = html.escape(text)
  return '<span style="%s">%s</span>' % ("; ".join(css), text) if css else text


def _note(text):
  return ('<div style="font-family: %s; font-size: 13px; opacity: .7; margin: 2px 0">%s</div>'
          % (_FONT, html.escape(text)))


def _duration(seconds):
  seconds = int(seconds)
  return "%dm %02ds" % (seconds // 60, seconds % 60) if seconds >= 60 else "%ds" % seconds


def _tail(text, lines=40):
  return "\n".join(text.strip().splitlines()[-lines:])


if get_ipython() is not None:
  get_ipython().register_magic_function(_sql_magic, magic_kind="cell", magic_name="sql")

_setup(run_on, project, bucket, region)

## 🔵 Shell 1: watch a table's changes

Create the source table, and open a live view of its changes. It stays open while you go on.

In [ ]:
%%sql shell1
CREATE EXTERNAL TABLE orders (
  id BIGINT PRIMARY KEY NOT ENFORCED,
  item VARCHAR,
  qty INTEGER
)
TYPE 'iceberg';

In [ ]:
%%sql shell1
SELECT * FROM CHANGES(orders)
WITH ('streaming' = true, 'starting_strategy' = 'earliest', 'poll_interval_seconds' = 1);

## 🟣 Shell 2: keep a second table in sync

Create the mirror table, and stream every change to `orders` into it. The two metadata columns give
the sink each change's type and order. On Dataflow, the job takes a few minutes to start.

In [ ]:
%%sql shell2
CREATE EXTERNAL TABLE orders_mirror (
  id BIGINT PRIMARY KEY NOT ENFORCED,
  item VARCHAR,
  qty INTEGER
)
TYPE 'iceberg';

In [ ]:
%%sql shell2
INSERT INTO orders_mirror
WITH ('mode' = 'merge-on-read',
      'change_type_column' = '_change_type',
      'triggering_frequency_seconds' = 1)
SELECT *, _change_type, _commit_snapshot_sequence_number
FROM CHANGES(orders)
WITH ('streaming' = true,
      'starting_strategy' = 'earliest',
      'poll_interval_seconds' = 1);

## 🟠 Shell 3: change the table

Write two rounds of changes. Shell 1's live view shows up next to each one, so you can watch the
changes arrive. After each round, both tables should hold the same rows.

In [ ]:
%%sql shell3
SHOW TABLES;

In [ ]:
%%sql shell3
-- Snapshot 1: three new orders
INSERT INTO orders
WITH (
  'mode' = 'merge-on-read',
  'change_type_column' = 'op',
  'sequence_number_column' = 'seq')
  (id, item, qty, op, seq)
VALUES (1, 'apple', 3, 'INSERT', 1),
       (2, 'pear', 5, 'INSERT', 1),
       (3, 'plum', 2, 'INSERT', 1);

In [ ]:
%%sql shell3
SELECT * FROM orders;
SELECT * FROM orders_mirror;

In [ ]:
%%sql shell3
-- Snapshot 2: change order 1, delete order 2, add order 4
INSERT INTO orders
WITH (
  'mode' = 'merge-on-read',
  'change_type_column' = 'op',
  'sequence_number_column' = 'seq')
  (id, item, qty, op, seq)
VALUES (1, 'apple', 3, 'UPDATE_BEFORE', 2),
       (1, 'apple', 4, 'UPDATE_AFTER', 2),
       (2, 'pear', 5, 'DELETE', 2),
       (4, 'kiwi', 1, 'INSERT', 2);

In [ ]:
%%sql shell3
SELECT * FROM orders;
SELECT * FROM orders_mirror;

## Clean up

Close the live view and stop the streaming pipeline, like pressing `q` in shell 1 and Ctrl+C in
shell 2. On Dataflow, this cancels the job, which otherwise keeps running.

In [ ]:
stop_all()